# XAI-IDS Benchmark - Kaggle Runner

Runs the full 4 (XAI) x 2 (DL) x 3 (dataset) factorial benchmark from the master's thesis proposal, on Kaggle.

**Kaggle setup (do these before running):**
1. **Settings -> Internet -> On** (needed to clone the GitHub repo).
2. **Settings -> Accelerator -> GPU** (P100 or T4 x2).
3. **Add Input datasets** (right panel -> Add Input -> search): add the three IDS datasets (search terms below). They mount under `/kaggle/input/<slug>/` - the notebook auto-detects them.
   - Search: `CIC-IDS-2017` or `cicids2017`
   - Search: `UNSW-NB15`
   - Search: `CIC IoT 2023` or `ciciot2023`

Modes:
- `--quick` (default): 2k rows, 20 epochs, ~1-3 hours on a P100. Use this first.
- Full run: set `QUICK = False`; for the 9h Kaggle limit, also set `MAX_EVAL` and `N_BOOTSTRAP` (suggested: `MAX_EVAL=500`, `N_BOOTSTRAP=30`).

Results are written to `/kaggle/working/results/` and auto-saved as notebook Output when you save the notebook.


## 1. Configuration

In [ ]:
GITHUB_REPO = 'https://github.com/SakiburRahman07/xai-ids-benchmark.git'
BRANCH = 'main'
QUICK = True                      # True = smoke run; False = full factorial
MAX_EVAL = None                    # None = default (500 quick / 2000 full); for 9h limit try 500
N_BOOTSTRAP = None                 # None = default (20 quick / 100 full); for 9h limit try 30
DATASETS = None                    # None = all 3 present; or e.g. ['cicids2017']
MODELS = None                      # None = both; or ['cnn1d','ft_transformer']
METHODS = None                     # None = all enabled; or ['shap','lime']
WORK_DIR = '/kaggle/working'        # Kaggle writable space
REPO_INPUT_SLUG = ''               # If you upload the repo zip as a Kaggle Dataset (Add Input), put its slug here (e.g., 'youruser/xai-ids-benchmark'). Empty = try git clone first.
print(f'Repo: {GITHUB_REPO} | Quick: {QUICK} | max_eval={MAX_EVAL} | n_boot={N_BOOTSTRAP}')

## 2. Get the repo and install dependencies

Two paths (auto-detected):
- **Internet On:** clones from GitHub.
- **Internet Off:** uses the repo uploaded as a Kaggle Dataset (set `REPO_INPUT_SLUG` below to the dataset slug you added via Add Input).

To get the repo into Kaggle without Internet: download the repo zip from GitHub on your laptop, then in Kaggle right panel → Add Input → New Dataset → upload the zip. Set `REPO_INPUT_SLUG` to the resulting slug.

In [ ]:
import os, subprocess, shutil, sys, glob
os.chdir(WORK_DIR)
REPO_DIR = os.path.join(WORK_DIR, 'xai-ids-benchmark')
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

# Path A: try git clone (needs Internet On).
cloned = False
try:
    subprocess.run(['git','clone','--depth','1','-b',BRANCH,GITHUB_REPO,REPO_DIR], check=True, capture_output=True)
    cloned = True
    print('Cloned from GitHub (Internet is On).')
except Exception as e:
    print('git clone failed (Internet likely Off):', e.stderr.decode() if hasattr(e,'stderr') and e.stderr else e)

# Path B: fall back to repo uploaded as a Kaggle Dataset.
if not cloned:
    KAGGLE_INPUT = '/kaggle/input'
    candidates = []
    if REPO_INPUT_SLUG:
        candidates.append(os.path.join(KAGGLE_INPUT, REPO_INPUT_SLUG))
    # Also scan all inputs for a run_benchmark.py file.
    if os.path.isdir(KAGGLE_INPUT):
        for slug in os.listdir(KAGGLE_INPUT):
            candidates.append(os.path.join(KAGGLE_INPUT, slug))
    src_root = None
    for c in candidates:
        if os.path.exists(os.path.join(c, 'scripts/run_benchmark.py')):
            src_root = c; break
        if os.path.exists(os.path.join(c, 'xai-ids-benchmark/scripts/run_benchmark.py')):
            src_root = os.path.join(c, 'xai-ids-benchmark'); break
    if src_root:
        # Copy (not symlink) so the working tree is writable.
        shutil.copytree(src_root, REPO_DIR)
        print(f'Copied repo from Kaggle Input: {src_root}')
    else:
        raise RuntimeError('No repo found. Enable Internet, or upload the repo zip as a Kaggle Dataset and set REPO_INPUT_SLUG.')

os.chdir(REPO_DIR)
if not os.path.exists('scripts/run_benchmark.py') and os.path.exists('xai-ids-benchmark/scripts/run_benchmark.py'):
    os.chdir('xai-ids-benchmark')
CODE_ROOT = os.getcwd()            # absolute path to the code dir (used by later import cells)
print('CWD / CODE_ROOT:', CODE_ROOT)
assert os.path.exists('scripts/run_benchmark.py'), 'run_benchmark.py not found'

In [ ]:
# Install deps not preinstalled on Kaggle. Kaggle has torch/numpy/pandas/scikit-learn.
# NOTE: rtdl is NOT needed - we use a local FT-Transformer implementation.
import importlib.util
def have(name): return importlib.util.find_spec(name) is not None
pkgs = ['shap','lime','dice_ml','captum','statsmodels','seaborn','pyyaml']
missing = [p for p in pkgs if not have(p)]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing])
print('deps OK')

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Link Kaggle Input datasets into `data_raw/`

Kaggle Input datasets mount under `/kaggle/input/...` (sometimes nested as `/kaggle/input/notebooks/<author>/<slug>/`). The cell below scans ALL of `/kaggle/input/` recursively for CSVs and classifies them by filename into the three target datasets, then symlinks them into `data_raw/<dataset>/` so the benchmark loaders find them.

If a dataset is missing, the benchmark will skip its cells gracefully.

In [ ]:
import os, glob, shutil
KAGGLE_INPUT = '/kaggle/input'
REPO_ROOT = os.getcwd()
print('REPO_ROOT:', REPO_ROOT)

# Recursively find ALL csvs under /kaggle/input (handles any nesting depth).
all_csvs = glob.glob(os.path.join(KAGGLE_INPUT, '**', '*.csv'), recursive=True)
print(f'Total CSVs found under {KAGGLE_INPUT}: {len(all_csvs)}')

# Classify by filename/path heuristic.
def classify(path):
    p = path.lower()
    name = os.path.basename(path).lower()
    # CICIDS2017: day-named CSVs, MachineLearningCSV folder, or known Kaggle slugs
    if (any(d in name for d in ['monday','tuesday','wednesday','thursday','friday'])
        or 'machinelerning' in p or 'cic-ids' in p or 'cicids' in p
        or 'network-intrusion' in p or 'workinghours' in name or 'pcap_iscx' in name):
        return 'cicids2017'
    # UNSW-NB15
    if 'unsw' in p and 'nb15' in p:
        return 'unsw_nb15'
    # CICIoT2023
    if 'ciciot' in p or 'cic-iot' in p or 'iot-2023' in p or 'iot2023' in p or 'unb-cic-iot' in p:
        return 'ciciot2023'
    return None

for ds in ['cicids2017','unsw_nb15','ciciot2023']:
    dst = os.path.join(REPO_ROOT, 'data_raw', ds)
    os.makedirs(dst, exist_ok=True)
    matched = [c for c in all_csvs if classify(c) == ds]
    for f in matched:
        link = os.path.join(dst, os.path.basename(f))
        if not os.path.exists(link):
            try:
                os.symlink(f, link)
            except OSError:
                shutil.copy(f, link)
    print(f'{ds}: linked {len(matched)} CSV(s) -> data_raw/{ds}/')
    for m in matched[:3]:
        print(f'  e.g. {m}')
    if len(matched) > 3:
        print(f'  ... and {len(matched)-3} more')

In [ ]:
# Verify datasets are present (prints manual instructions if not).
!python scripts/prepare_data.py --base . || echo 'Add the dataset via the right panel -> Add Input.'

## 4. Run the benchmark

Smoke run uses 2k rows / 20 epochs (1-3 hours on P100). For the full factorial within the 9h Kaggle limit, set `QUICK=False`, `MAX_EVAL=500`, `N_BOOTSTRAP=30`.

In [ ]:
OUT = os.path.join(WORK_DIR, 'results')
args = [sys.executable, 'scripts/run_benchmark.py', '--config-dir','config', '--base','.', '--out-dir', OUT]
if QUICK: args.append('--quick')
if MAX_EVAL:     args += ['--max-eval', str(MAX_EVAL)]
if N_BOOTSTRAP:  args += ['--n-bootstrap', str(N_BOOTSTRAP)]
if DATASETS:     args += ['--datasets', *DATASETS]
if MODELS:       args += ['--models', *MODELS]
if METHODS:      args += ['--methods', *METHODS]
print('Running:', ' '.join(args))
subprocess.run(args, check=False)

## 5. Results & analysis (Friedman/Nemenyi, H2 Wilcoxon)

In [ ]:
import pandas as pd, json, os
tidy_path = os.path.join(OUT, 'tidy_metrics.csv')
if os.path.exists(tidy_path):
    tidy = pd.read_csv(tidy_path)
    print('Tidy metric rows:', len(tidy))
    print(tidy.head(20))
else:
    print('No tidy_metrics.csv - did the run produce results?')

In [ ]:
fn_path = os.path.join(OUT, 'friedman_nemenyi.json')
if os.path.exists(fn_path):
    with open(fn_path) as f:
        fn = json.load(f)
    for metric, res in fn.items():
        print(f'\n=== {metric} ===')
        print('avg ranks:', res.get('avg_ranks'))
        print('CD:', round(res.get('critical_distance',0),3), '| p:', res.get('p_value'))
        for pair, dec in res.get('pairwise',{}).items():
            print('  ', pair, 'diff=', round(dec['rank_diff'],3), 'sig=', dec['significant'])
else:
    print('No friedman_nemenyi.json - insufficient data.')

In [ ]:
# Critical distance diagram for a chosen metric.
import sys, os
# Robustly locate src/ (CWD may have drifted between cells).
for base in [os.getcwd(), '/kaggle/working', '/kaggle/working/xai-ids-benchmark', '/kaggle/working/xai-ids-benchmark/xai-ids-benchmark']:
    cand = os.path.join(base, 'src')
    if os.path.exists(os.path.join(cand, 'xai_ids_benchmark', '__init__.py')):
        sys.path.insert(0, cand); print('using src:', cand); break
from xai_ids_benchmark.analysis.plots import plot_cd
metric_to_plot = 'deletion_auc'
if os.path.exists(fn_path) and metric_to_plot in fn:
    plot_cd(fn[metric_to_plot]['avg_ranks'], fn[metric_to_plot]['critical_distance'],
            output_path=os.path.join(OUT,'cd_diagram.png'), title=f'CD diagram - {metric_to_plot}')
    from IPython.display import Image, display
    display(Image(os.path.join(OUT,'cd_diagram.png')))
else:
    print('CD diagram not produced. Available metrics:', list(fn.keys()) if os.path.exists(fn_path) else 'no results file')

In [ ]:
# H2: stability degradation on UNSW-NB15 (Wilcoxon).
h2_path = os.path.join(OUT, 'h2_wilcoxon.json')
if os.path.exists(h2_path):
    with open(h2_path) as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print('H2 file not produced (insufficient stability data).')

## 6. Diagnostics

If `results/` is empty, the benchmark skipped all cells — usually because the datasets weren't linked into `data_raw/` (Kaggle Input not added). This cell shows the state.

In [ ]:
import os, glob
print('=== /kaggle/input/ (Kaggle Input datasets you added) ===')
if os.path.isdir('/kaggle/input'):
    for slug in os.listdir('/kaggle/input'):
        sub = os.path.join('/kaggle/input', slug)
        n_csv = len(glob.glob(os.path.join(sub, '**', '*.csv'), recursive=True))
        print(f'  {slug}: {n_csv} CSV file(s)')
else:
    print('  /kaggle/input does not exist - no Input datasets added.')
print('\n=== data_raw/ (linked datasets) ===')
for ds in ['cicids2017','unsw_nb15','ciciot2023']:
    d = os.path.join(os.getcwd(), 'data_raw', ds)
    n = len(glob.glob(os.path.join(d, '*.csv'))) if os.path.isdir(d) else 0
    print(f'  data_raw/{ds}/: {n} CSV file(s) -> {"OK" if n>0 else "EMPTY - dataset not linked"}')
print('\n=== results/ ===')
if os.path.isdir(OUT):
    for f in sorted(os.listdir(OUT)):
        print(f'  {f}')
else:
    print(f'  {OUT} does not exist - benchmark produced no output.')
    print('\n  => Most likely cause: no datasets in /kaggle/input/.')
    print('     Fix: right panel -> Add Input -> search CIC-IDS-2017, UNSW-NB15, CIC IoT 2023.')
    print('     Then re-run the dataset-linking cell (section 3) and the benchmark cell (section 4).')

## 7. Save results

Results in `/kaggle/working/results/` are auto-saved as notebook Output when you Save the notebook. You can also download them directly.

In [ ]:
import shutil, os
if not os.path.isdir(OUT):
    print('No results/ folder to zip. Run the diagnostic cell above to see why the benchmark produced no output.')
else:
    zip_path = os.path.join(WORK_DIR, 'results.zip')
    shutil.make_archive(zip_path.replace('.zip',''), 'zip', OUT)
    print('Saved:', zip_path)